# Lab — Real Dataset Bench: Hugging Face + Kaggle

This is the real-data replacement for toy corpus examples. Students pull small/streaming slices from multiple public datasets, inspect schemas, normalize them, measure them, filter/deduplicate them, and build competing mixtures.

Use the registry in data/REAL_DATASET_REGISTRY.md. Do not vendor large third-party corpora into the repository.

Learning loop: predict → acquire → inspect → normalize → measure → filter → deduplicate → mix → report → train.

In [ ]:
!pip -q install -U datasets huggingface_hub pandas pyarrow kagglehub
import json, re, hashlib, pathlib, statistics, random
from collections import Counter, defaultdict
from datasets import load_dataset
print('ready')

## 1. Real sources

Hugging Face: FineWeb, FineWeb-Edu, Dolma, Wikipedia, OpenWebMath, Aya, OpenR1-Math-220k.

Multilingual case study: CulturaX (gated; accept current access conditions before use).

Kaggle: Tashkeela Clean Arabic, RUFND Roman-Urdu, Gita Verses.

See data/REAL_DATASET_REGISTRY.md for the exact links, dataset IDs, scales, fields, licenses, and roles.

In [ ]:
N = 100

HF_SOURCES = [
    {'name':'fineweb','id':'HuggingFaceFW/fineweb','config':'sample-10BT'},
    {'name':'fineweb_edu','id':'HuggingFaceFW/fineweb-edu','config':'sample-10BT'},
    {'name':'wikipedia_en','id':'wikimedia/wikipedia','config':'20231101.en'},
    {'name':'openwebmath','id':'open-web-math/open-web-math','config':None},
    {'name':'aya','id':'CohereLabs/aya_dataset','config':None},
    {'name':'openr1_math','id':'open-r1/OpenR1-Math-220k','config':'default'},
]

# Persian-first sources are kept in the same bench so the learner can compare
# language, source quality, provenance, and scale without creating another notebook.
PERSIAN_SOURCES = [
    {'name':'persian_synthetic','id':'ibnsina-llm/synthetic-persian-v1','config':None},
    {'name':'persian_culturax','id':'uonlp/CulturaX','config':'fa'},
    {'name':'persian_fineweb2_hq','id':'epfml/FineWeb2-HQ','config':'fas_Arab'},
    {'name':'persian_naab','id':'SLPL/naab','config':None},
    {'name':'persian_merged','id':'PersianML/persian-text-corpus','config':None},
]

ENABLE_PERSIAN = True

if ENABLE_PERSIAN:
    HF_SOURCES.extend(PERSIAN_SOURCES);
def stream_sample(spec, n=N):
    kwargs={'split':'train','streaming':True}
    if spec['config'] is not None:
        kwargs['name']=spec['config']
    ds=load_dataset(spec['id'], **kwargs)
    return [row for i,row in zip(range(n),ds)]

samples={}
for spec in HF_SOURCES:
    try:
        samples[spec['name']]=stream_sample(spec)
        print(spec['name'],len(samples[spec['name']]))
    except Exception as e:
        print('SKIP',spec['name'],type(e).__name__,str(e)[:160])

## Persian longitudinal track

Persian is treated as a first-class language case study, not a single dataset. The **1.5B** item commonly associated with this track is the **IbnSina-1.5B model**, not a 1.5B-token corpus. The bench samples Persian corpora alongside the general sources above.

Compare at minimum: **IbnSina Synthetic Persian v1**, **CulturaX-fa**, **FineWeb2-HQ fas_Arab**, **Naab**, and the **Persian Corpus (Merged)**. Do not treat the largest corpus as automatically best.

Also keep **PersianMedQA** evaluation-only and access-gated; it belongs in the evaluation lesson, not the training mixture.


## 2. Inspect real schemas

Do not guess column names. Print the first record from every source before writing normalization code.

In [ ]:
for name, rows in samples.items():
    if not rows: continue
    print('\n###',name)
    print('fields:',list(rows[0].keys()))
    for key,value in rows[0].items():
        print(key,':',str(value).replace('\n',' ')[:220])

## 3. Normalize into a common training schema

Canonical fields:

source_id
record_id
language
text
prompt
target
metadata_json

This is the real data-engineering step: downstream training code should not need to know every source's original schema.

In [ ]:
def normalize(name,row,idx):
    text=''
    prompt=''
    target=''
    language=None
    record_id=str(row.get('id',idx))

    if name in {'fineweb','fineweb_edu','openwebmath','persian_synthetic','persian_culturax','persian_fineweb2_hq','persian_naab','persian_merged'}:
        text=str(row.get('text',''))
        language=row.get('language') or ('fa' if name.startswith('persian_') else None)
    elif name=='wikipedia_en':
        text=str(row.get('text',''))
        language='en'
    elif name=='aya':
        prompt=str(row.get('inputs',''))
        target=str(row.get('targets',''))
        text=prompt+'\n'+target
        language=row.get('language') or row.get('language_code')
        record_id=str(row.get('user_id',idx))
    elif name=='openr1_math':
        prompt=str(row.get('problem',''))
        target=str(row.get('solution',''))
        text=prompt+'\n'+target
        language='en'
        record_id=str(row.get('uuid',idx))

    metadata={k:v for k,v in row.items() if k not in {'text','inputs','targets','problem','solution'}}
    return {
        'source_id':name,
        'record_id':record_id,
        'language':language,
        'text':text,
        'prompt':prompt,
        'target':target,
        'metadata_json':json.dumps(metadata,default=str)
    }

canonical=[normalize(name,row,i) for name,rows in samples.items() for i,row in enumerate(rows)]
print('canonical rows:',len(canonical))

## 4. Measure source characteristics

Whitespace words are only a diagnostic. Later rerun the exact same analysis with the model tokenizer and report token fertility.

In [ ]:
def stats(rows):
    lengths=[len(r['text']) for r in rows if r['text']]
    words=[len(r['text'].split()) for r in rows if r['text']]
    return {
        'rows':len(rows),
        'mean_chars':round(statistics.mean(lengths),1) if lengths else 0,
        'p95_chars':sorted(lengths)[max(0,int(len(lengths)*.95)-1)] if lengths else 0,
        'mean_words':round(statistics.mean(words),1) if words else 0,
        'languages':Counter(str(r['language']) for r in rows)
    }

for source in sorted(set(r['source_id'] for r in canonical)):
    print(source,stats([r for r in canonical if r['source_id']==source]))

## 5. Filter and exact-deduplicate

Start with transparent heuristics. A heuristic is not a validated quality model.

In [ ]:
def repeat_ratio(text):
    if not text: return 1.0
    counts=Counter(text)
    return max(counts.values())/len(text)

def keep(row):
    t=row['text'].strip()
    if len(t)<40: return False
    if len(t)>2_000_000: return False
    if repeat_ratio(t)>.60: return False
    if row['prompt'] and not row['target'].strip(): return False
    return True

filtered=[r for r in canonical if keep(r)]

def norm_text(text):
    return re.sub(r'\s+',' ',text.strip().lower())

seen={}
dedup=[]
cross_source=[]
for r in filtered:
    h=hashlib.sha256(norm_text(r['text']).encode()).hexdigest()
    if h in seen:
        if seen[h]['source_id']!=r['source_id']:
            cross_source.append((seen[h]['source_id'],r['source_id']))
        continue
    seen[h]=r
    dedup.append(r)

print('canonical:',len(canonical))
print('filtered:',len(filtered))
print('deduped:',len(dedup))
print('cross-source duplicate pairs:',len(cross_source))

## 6. Build competing real-data mixtures

Mixture A emphasizes educational and specialist content. Mixture B is more general-web-heavy.

Hold the total token budget constant in downstream model training.

In [ ]:
groups=defaultdict(list)
for r in dedup:
    groups[r['source_id']].append(r)

mix_a={'fineweb':.50,'fineweb_edu':.20,'wikipedia_en':.10,'openwebmath':.10,'aya':.10}
mix_b={'fineweb':.70,'fineweb_edu':.10,'wikipedia_en':.10,'openwebmath':.05,'aya':.05}

def make_mixture(weights,n=500,seed=42):
    rng=random.Random(seed)
    out=[]
    for source,weight in weights.items():
        pool=groups.get(source,[])
        k=round(n*weight)
        if pool:
            out.extend(rng.sample(pool,min(k,len(pool))))
    rng.shuffle(out)
    return out

A=make_mixture(mix_a)
B=make_mixture(mix_b)
print('A:',Counter(r['source_id'] for r in A))
print('B:',Counter(r['source_id'] for r in B))

## 7. Kaggle: acquire real files

Configure Kaggle authentication in the runtime; never commit a Kaggle token.

Datasets:

Tashkeela — ahmedmohsen2002/tashkeela-clean-arabic-diacritized-corpus

RUFND — zainabnoor02/rufnd-cleaned-testtrain-data-files

Gita Verses — ashishk1331/gita-verses

Keep these sources separate from pretraining mixtures unless the experiment explicitly calls for them. They are useful for Arabic, Roman-Urdu, and multilingual schema/language studies.

In [ ]:
ENABLE_KAGGLE=False

KAGGLE_IDS={
    'tashkeela':'ahmedmohsen2002/tashkeela-clean-arabic-diacritized-corpus',
    'rufnd':'zainabnoor02/rufnd-cleaned-testtrain-data-files',
    'gita':'ashishk1331/gita-verses'
}

if ENABLE_KAGGLE:
    import kagglehub
    kaggle_paths={}
    for name,dataset_id in KAGGLE_IDS.items():
        try:
            kaggle_paths[name]=kagglehub.dataset_download(dataset_id)
            print(name,kaggle_paths[name])
        except Exception as e:
            print('SKIP',name,str(e)[:160])
else:
    print('Kaggle disabled; set ENABLE_KAGGLE=True after authentication.')

In [ ]:
if ENABLE_KAGGLE:
    for name,root in kaggle_paths.items():
        print('\n###',name)
        for path in pathlib.Path(root).rglob('*'):
            if path.is_file() and path.suffix.lower() in {'.csv','.txt','.parquet'}:
                print(path.name,round(path.stat().st_size/1e6,2),'MB')

## 8. Model-facing experiments

### Experiment A — source mixture

Train the same small model on A and B. Hold model, tokenizer, optimizer, sequence length, and total training tokens fixed.

Measure target quality, validation loss, language slices, and resource usage.

### Experiment B — quality filtering

Compare no filter versus the transparent filter. Keep source weights fixed.

### Experiment C — multilingual exposure

Change only the language/source allocation and keep total tokens fixed.

### Experiment D — source overlap

Measure exact overlap between sources before training. For benchmark data, add near-duplicate and temporal contamination checks.

### Experiment E — pretraining versus SFT

Do not blindly concatenate Aya or OpenR1 reasoning/instruction records into a causal pretraining corpus. Put them into an explicit SFT/reasoning experiment and compare the appropriate objective.

## 9. Required artifact

Export:

dataset manifest + source/version records + schema map + before/after statistics + dedup report + mixture definitions + sample records + license/access notes + experiment card.

Save the machine-readable manifest from this notebook to /content/real_dataset_bench_manifest.json.

In [ ]:
manifest={
    'sources_seen':sorted(set(r['source_id'] for r in canonical)),
    'rows_seen':len(canonical),
    'rows_after_filter':len(filtered),
    'rows_after_exact_dedup':len(dedup),
    'cross_source_duplicate_pairs':len(cross_source),
    'mixture_A':mix_a,
    'mixture_B':mix_b,
}
out='/content/real_dataset_bench_manifest.json'
with open(out,'w') as f:
    json.dump(manifest,f,indent=2)
print(out)
print(json.dumps(manifest,indent=2))

## Persian experiment contract

Run the same controlled experiments for Persian: tokenizer fertility, filtering, exact/near deduplication, source mixing, and real-vs-synthetic ablation. Hold model, tokenizer, token budget, and evaluation protocol fixed when comparing interventions.

Required evidence: source revision/access record, schema map, before/after statistics, dedup report, mixture definition, Persian evaluation results, multilingual regression results, and an experiment card.
